# Impacts of Over Sampling

# ⚠️ Environment Requirements

This notebook requires a specific conda environment to run correctly.

## Setup

```bash
conda create -n timegan_env python=3.11
conda activate timegan_env
pip install tensorflow==2.15.0
pip install numpy==2.0.0
pip install torch==2.1.0
pip install scikit-learn imbalanced-learn jupyter ipykernel
pip install sktime==0.21.0
pip install numba==0.59.0
pip install matplotlib==3.8.0
python -m ipykernel install --user --name timegan_env --display-name "timegan_env"
```

## Then in Jupyter
- Go to **Kernel → Change Kernel → timegan_env**

## Notes
- `tensorflow==2.15.0` — last version before Keras 3, required for TimeGAN (`MultiRNNCell` support)
- `numpy==2.0.0` — compatible with both TF 2.15 and pickle files saved with numpy 2.x
- `torch==2.1.0` — for GRU experiments
- `sktime==0.21.0` + `numba==0.59.0` — required for Catch22 feature extraction
- `matplotlib==3.8.0` — for t-SNE and other visualizations

### TimeGAN Training

In [1]:
import pickle
import numpy as np

# ══════════════════════════════════════════════════════════════
# LOAD SAVED ENN-CLEANED HYBRID DATA
# Input folder: ./final_split_data_HybridNorm_ENN
# Output variables:
#   X_train, y_train
#   X_val,   y_val
#   X_test,  y_test
# ══════════════════════════════════════════════════════════════

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]


ENN_DIR = "./final_split_data_HybridNorm_ENN"

X_train, y_train = load_split(f"{ENN_DIR}/train_set.pkl")
X_val,   y_val   = load_split(f"{ENN_DIR}/val_set.pkl")
X_test,  y_test  = load_split(f"{ENN_DIR}/test_set.pkl")

print("Loaded ENN-cleaned Hybrid dataset:")
print(f"Train: X={X_train.shape} | y={y_train.shape}")
print(f"Val  : X={X_val.shape} | y={y_val.shape}")
print(f"Test : X={X_test.shape} | y={y_test.shape}")

print("\nClass counts:")
print("Train:", np.bincount(y_train.astype(int), minlength=2))
print("Val  :", np.bincount(y_val.astype(int), minlength=2))
print("Test :", np.bincount(y_test.astype(int), minlength=2))

Loaded ENN-cleaned Hybrid dataset:
Train: X=(12268, 288, 10) | y=(12268,)
Val  : X=(1780, 288, 10) | y=(1780,)
Test : X=(3559, 288, 10) | y=(3559,)

Class counts:
Train: [12150   118]
Val  : [1763   17]
Test : [3525   34]


In [2]:
import numpy as np
import pickle
import time
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from timegan import timegan

# ══════════════════════════════════════════════════════════════
# TIMEGAN FINAL DATASET GENERATION — ENN-CLEANED HYBRID DATA
# Creates:
#   timegan_balanced  -> no RUS, all Non-SEP, SEP matched to all Non-SEP
#   timegan_8000
#   timegan_4000
#   timegan_2000
#   timegan_1000
#   timegan_500
# ══════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════
# STEP 1 — PREPARE REAL SEP AND NON-SEP DATA
# ══════════════════════════════════════════════════════════════
# Assumes ENN-cleaned data already loaded:
# X_train, y_train, X_val, y_val, X_test, y_test

y_int = y_train.astype(int)

idx_real_sep = np.where(y_int == 1)[0]
idx_nsep     = np.where(y_int == 0)[0]

n_sep_real = len(idx_real_sep)

X_real_sep = X_train[idx_real_sep].astype(np.float32)
X_nsep_all = X_train[idx_nsep].astype(np.float32)

print(f"\n  Real SEP samples     : {n_sep_real}")
print(f"  Real Non-SEP samples : {len(X_nsep_all)}")
print(f"  SEP shape            : {X_real_sep.shape}")

ori_data = X_real_sep.astype(np.float32)


# ══════════════════════════════════════════════════════════════
# STEP 2 — TRAIN TIMEGAN AND GENERATE ENOUGH SYNTHETIC SEP
# ══════════════════════════════════════════════════════════════

parameters = {
    "module": "gru",
    "hidden_dim": 72,
    "num_layer": 3,
    "iterations": 8000,
    "batch_size": 128,
}

# Need enough synthetic SEP for:
# 1) timegan_balanced: SEP must match all Non-SEP
# 2) timegan_8000: SEP must reach 8000
n_generate_timegan = max(
    8000 - n_sep_real,
    len(X_nsep_all) - n_sep_real
)

if n_generate_timegan <= 0:
    raise ValueError(
        "No synthetic SEP samples are needed. "
        "Check SEP and Non-SEP counts."
    )

print(f"\n  Training TimeGAN on {n_sep_real} real SEP samples …")
print(f"  Synthetic SEP samples to generate: {n_generate_timegan}")

generated_data, total_train_time, infer_time = timegan(
    ori_data,
    parameters,
    n_generate=n_generate_timegan
)

X_synthetic_all = np.array(generated_data).astype(np.float32)
n_synthetic = len(X_synthetic_all)

print(f"\n  ✓ Total training time                    : {total_train_time:.2f}s")
print(f"  ✓ Inference time ({n_synthetic} samples) : {infer_time:.4f}s")
print(f"  ✓ Time per sample                        : {infer_time / n_synthetic * 1000:.4f}ms")
print(f"  ✓ Generated shape                        : {X_synthetic_all.shape}")


# ══════════════════════════════════════════════════════════════
# STEP 3 — SAVE REAL VS SYNTHETIC SEP SAMPLES
# ══════════════════════════════════════════════════════════════

rng = np.random.default_rng(42)

n_compare = min(118, len(X_real_sep), len(X_synthetic_all))

idx_real_compare = rng.choice(
    len(X_real_sep),
    size=n_compare,
    replace=False
)

idx_synthetic_compare = rng.choice(
    len(X_synthetic_all),
    size=n_compare,
    replace=False
)

SAVE_DIR = "./sep_samples"
os.makedirs(SAVE_DIR, exist_ok=True)

save_path = os.path.join(SAVE_DIR, "sep_real_vs_synthetic_timegan.pkl")

with open(save_path, "wb") as f:
    pickle.dump({
        "X_real_sep": X_real_sep[idx_real_compare],
        "y_real_sep": np.zeros(n_compare, dtype=np.int32),      # 0 = real

        "X_synthetic_sep": X_synthetic_all[idx_synthetic_compare],
        "y_synthetic_sep": np.ones(n_compare, dtype=np.int32),  # 1 = synthetic

        "total_train_time": total_train_time,
        "infer_time": infer_time,
        "n_sep_real": n_sep_real,
        "n_synthetic": n_synthetic,
        "n_compare": n_compare,
    }, f)

print(f"\n✅ Saved real vs synthetic ({n_compare} each) → {os.path.abspath(save_path)}")
print(f"   real SEP      : {n_compare} samples  (label=0)")
print(f"   synthetic SEP : {n_compare} samples  (label=1)")


# ══════════════════════════════════════════════════════════════
# STEP 4 — BUILD DATASETS FROM SYNTHETIC SEP
# ══════════════════════════════════════════════════════════════

print(f"\n  Available Non-SEP samples : {len(X_nsep_all)}")
print(f"  Real SEP samples          : {n_sep_real}")
print(f"  Synthetic SEP samples     : {n_synthetic}")

sep_configs = [
    ("timegan_balanced", len(X_nsep_all)),   # no RUS
    ("timegan_8000",    8000),
    ("timegan_4000",    4000),
    ("timegan_2000",    2000),
    ("timegan_1000",    1000),
    ("timegan_500",      500),
]

timegan_datasets = {}

print(
    f"\n{'Variant':<20} "
    f"{'NSEP':>8} "
    f"{'SEP':>8} "
    f"{'Real':>8} "
    f"{'Synthetic':>10} "
    f"{'RUS?':>8} "
    f"{'Shape'}"
)
print("─" * 90)

for cfg_name, n_sep in sep_configs:

    if n_sep > len(X_nsep_all):
        raise ValueError(
            f"{cfg_name}: requested {n_sep} Non-SEP samples, "
            f"but only {len(X_nsep_all)} are available."
        )

    if n_sep < n_sep_real:
        raise ValueError(
            f"{cfg_name}: target SEP count {n_sep} is smaller than "
            f"the number of real SEP samples {n_sep_real}."
        )

    n_synthetic_needed = n_sep - n_sep_real

    if n_synthetic_needed > len(X_synthetic_all):
        raise ValueError(
            f"{cfg_name}: requested {n_synthetic_needed} synthetic SEP samples, "
            f"but only {len(X_synthetic_all)} are available."
        )

    # -----------------------------------------------------
    # Select synthetic SEP
    # -----------------------------------------------------
    idx_syn = rng.choice(
        len(X_synthetic_all),
        size=n_synthetic_needed,
        replace=False
    )

    X_syn = X_synthetic_all[idx_syn]

    # Combine all real SEP + selected synthetic SEP
    X_sep = np.concatenate([X_real_sep, X_syn], axis=0).astype(np.float32)
    y_sep = np.ones(len(X_sep), dtype=np.float32)

    # -----------------------------------------------------
    # Select Non-SEP
    # timegan_balanced uses all Non-SEP, no RUS
    # other variants randomly select n_sep Non-SEP
    # -----------------------------------------------------
    if cfg_name == "timegan_balanced":
        X_ns = X_nsep_all.copy()
        use_rus = False
    else:
        idx_ns = rng.choice(
            len(X_nsep_all),
            size=n_sep,
            replace=False
        )
        X_ns = X_nsep_all[idx_ns]
        use_rus = True

    y_ns = np.zeros(len(X_ns), dtype=np.float32)

    # -----------------------------------------------------
    # Combine and shuffle
    # -----------------------------------------------------
    X_tr = np.concatenate([X_ns, X_sep], axis=0).astype(np.float32)
    y_tr = np.concatenate([y_ns, y_sep], axis=0)

    idx_shuf = rng.permutation(len(X_tr))
    X_tr = X_tr[idx_shuf]
    y_tr = y_tr[idx_shuf]

    print(
        f"{cfg_name:<20} "
        f"{len(X_ns):>8} "
        f"{len(X_sep):>8} "
        f"{n_sep_real:>8} "
        f"{n_synthetic_needed:>10} "
        f"{str(use_rus):>8} "
        f"{X_tr.shape}"
    )

    timegan_datasets[cfg_name] = {
        "X_train": X_tr,
        "y_train": y_tr,

        "X_val": X_val,
        "y_val": y_val,

        "X_test": X_test,
        "y_test": y_test,

        "real_sep": n_sep_real,
        "synthetic_sep": n_synthetic_needed,
        "nsep": len(X_ns),
        "sep": len(X_sep),
        "use_rus": use_rus,
    }


# ══════════════════════════════════════════════════════════════
# STEP 5 — SAVE ALL TIMEGAN DATASETS
# ══════════════════════════════════════════════════════════════

DATASET_SAVE_DIR = "./timegan_datasets"
os.makedirs(DATASET_SAVE_DIR, exist_ok=True)

for cfg_name, d in timegan_datasets.items():

    save_path = os.path.join(DATASET_SAVE_DIR, f"{cfg_name}.pkl")

    with open(save_path, "wb") as f:
        pickle.dump(d, f)

    print(f"Saved {cfg_name} → {save_path}")

print(f"\n✅ All TimeGAN datasets saved to {os.path.abspath(DATASET_SAVE_DIR)}/")
print(f"   Total training time                  : {total_train_time:.2f}s")
print(f"   Inference time ({n_synthetic} samples): {infer_time:.4f}s")
print(f"   Time per sample                      : {infer_time / n_synthetic * 1000:.4f}ms")

print("\nDatasets created:")
for key in timegan_datasets.keys():
    print(" ", key)


  Real SEP samples     : 118
  Real Non-SEP samples : 12150
  SEP shape            : (118, 288, 10)

  Training TimeGAN on 118 real SEP samples …
  Synthetic SEP samples to generate: 12032
Start Embedding Network Training
step: 0/8000, e_loss: 0.3698
step: 1000/8000, e_loss: 0.0162
step: 2000/8000, e_loss: 0.0114
step: 3000/8000, e_loss: 0.0082
step: 4000/8000, e_loss: 0.0069
step: 5000/8000, e_loss: 0.0062
step: 6000/8000, e_loss: 0.0048
step: 7000/8000, e_loss: 0.0041
Finish Embedding Network Training
Start Training with Supervised Loss Only
step: 0/8000, s_loss: 0.1798
step: 1000/8000, s_loss: 0.0252
step: 2000/8000, s_loss: 0.0241
step: 3000/8000, s_loss: 0.0238
step: 4000/8000, s_loss: 0.0238
step: 5000/8000, s_loss: 0.0231
step: 6000/8000, s_loss: 0.0229
step: 7000/8000, s_loss: 0.0224
Finish Training with Supervised Loss Only
Start Joint Training
step: 0/8000, d_loss: 1.8975, g_loss_u: 1.058, g_loss_s: 0.0518, g_loss_v: 0.196, e_loss_t0: 0.0744
step: 1000/8000, d_loss: 2.8367, 